# 과제1 · Movie Agent

완전한 에이전트 루프를 갖춘 영화 에이전트입니다.

1. 영화에 대한 사용자 질문을 받습니다.
2. 모델이 OpenAI `tools` 파라미터로 **어떤 도구를 호출할지 스스로 결정**합니다.
3. 우리가 **실제 API를 호출**하고 결과를 `role: "tool"` 메시지로 모델에 돌려줍니다.
4. 모델이 도구 없이 최종 답변을 줄 때까지 **루프를 반복**합니다.

전체 대화 기록(`messages`)을 계속 누적해 **멀티턴 메모리**를 지원합니다.

In [ ]:
import json
import os

import httpx
from openai import OpenAI
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY 가 .env 에 설정되어 있지 않습니다."

client = OpenAI()
MODEL = "gpt-4o-mini"
BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"

In [ ]:
# --- 에이전트가 실제로 호출할 3개의 도구 (실제 API 호출) ---


def get_popular_movies():
    """/movies - 현재 인기 영화 목록"""
    return httpx.get(f"{BASE_URL}/movies", timeout=20).json()


def get_movie_details(id):
    """/movies/:id - 영화 상세 정보"""
    return httpx.get(f"{BASE_URL}/movies/{id}", timeout=20).json()


def get_similar_movies(id):
    """/movies/:id/similar - 비슷한 영화 목록"""
    return httpx.get(f"{BASE_URL}/movies/{id}/similar", timeout=20).json()


# 함수 이름 -> 실제 파이썬 함수 매핑
FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_similar_movies": get_similar_movies,
}

# OpenAI tools 파라미터로 넘길 도구 스키마
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "현재 인기 있는 영화 목록을 가져옵니다. 인자가 필요 없습니다.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "특정 영화의 상세 정보(줄거리, 장르, 평점 등)를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer", "description": "영화의 고유 ID"},
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "특정 영화와 비슷한 영화 목록을 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer", "description": "기준이 되는 영화의 고유 ID"},
                },
                "required": ["id"],
            },
        },
    },
]

In [ ]:
# --- 에이전트 루프 + 멀티턴 메모리 ---
# messages 리스트 전체를 매 턴 모델에 전달하는 것이 곧 "메모리"입니다.

messages = [
    {
        "role": "system",
        "content": (
            "당신은 친절한 한국어 영화 에이전트입니다. 필요하면 도구를 사용하세요. "
            "이전 대화에 나온 영화 id를 기억해 후속 질문에 활용하세요."
        ),
    }
]


def call_ai():
    """모델을 호출하고 응답을 처리합니다."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
    )
    process_ai_response(response.choices[0].message)


def process_ai_response(message):
    # 도구 호출이 없으면 -> 최종 답변
    if not message.tool_calls:
        messages.append({"role": "assistant", "content": message.content})
        print(f"Agent: {message.content}\n")
        return

    # 도구 호출이 있으면 -> assistant 메시지를 기록한 뒤 각 도구를 실제 실행
    messages.append(message)
    for tool_call in message.tool_calls:
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments or "{}")
        print(f"  [도구 호출] {name}({args})")

        result = FUNCTION_MAP[name](**args)

        messages.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result, ensure_ascii=False),
            }
        )

    # 결과를 모델에 다시 전달 -> 최종 답변까지 루프 반복
    call_ai()


def chat(user_input):
    print(f"User: {user_input}")
    messages.append({"role": "user", "content": user_input})
    call_ai()

In [ ]:
# --- 예시 상호작용: 메모리를 활용한 멀티턴 대화 ---

chat("지금 인기 있는 영화 알려줘")
chat("듄에 대해 더 알려줘")        # 위 목록에서 '듄'의 id를 기억해 상세 조회
chat("비슷한 영화 추천해 줄래?")    # 직전 영화 id로 비슷한 영화 조회